In [1]:
import pandas as pd

In [2]:
dataset_fps = ['aneesh_labels.csv', 'rishaant_labels.csv', 'josh_labels.csv', 'sharon_labels.csv']
dataset_fps = ['camera_data/' + fp for fp in dataset_fps]

replace_dataset = 'camera_data/majority_replaced_labels.csv'

In [3]:
def get_most_common_choice(dataset_fps, column, replace_fp):
    datasets = [pd.read_csv(dataset_fp) for dataset_fp in dataset_fps]
    column_choices = pd.concat([dataset[column] for dataset in datasets], axis = 1)

    most_common_choices = column_choices.mode(axis = 1)[0]

    assert len(datasets[0]) == len(most_common_choices)

    return_dataset = datasets[0].copy()

    return_dataset[column] = most_common_choices

    replacement_choices = pd.read_csv(replace_fp)[column]

    replacement_choices = replacement_choices[replacement_choices.notna()]

    return_dataset.loc[replacement_choices.index, column] = replacement_choices


    assert len(return_dataset) == len(datasets[0])

    return return_dataset


In [4]:
import requests
datasets = [
    "/sanitized-datasets/3c72418d-4248-5130-952c-aa43f87a7a8f", #Coronado Hills data
    "/sanitized-datasets/a405359a-1619-513b-b7ef-a472e3d4f131" #All data
  ]

data = []
for dataset in datasets:
    request = requests.get(f'https://tools.alertcalifornia.org{dataset}').json()
    data.append(pd.DataFrame(request['frames']))

get_id = lambda url: url.split('/')[-1]

In [5]:
data[1]['camera_id'].value_counts()

camera_id
Axis-MesaGrandeNorth    3113
Axis-CoronadoHillsS     2334
Axis-PalomarObs1        2044
Axis-ToroPeak1           512
Axis-Dewdrop1            424
Axis-Berryessa           404
Name: count, dtype: int64

In [ ]:
majority_labels = get_most_common_choice(dataset_fps, 'choice', replace_dataset)

majority_labels.to_csv('camera_data/majority_labels.csv', index = False)

coronado_urls = data[1][data[1]['url'].apply(get_id).isin(data[0]['url'].apply(get_id))]['url'].apply(lambda x: 'https://tools.alertcalifornia.org' + x)


coronado_labels = majority_labels[majority_labels['image'].isin(coronado_urls)]

coronado_labels_to_inject = coronado_labels[['image', 'choice']]
coronado_labels_to_inject['image'] = coronado_labels_to_inject['image'].apply(get_id)

coronado_labels_to_inject.columns = ['id', 'choice']

coronado_labels_to_inject.to_csv('camera_data/coronado_labels_to_inject.csv', index = False)
coronado_labels.to_csv('camera_data/coronado_hills_data.csv', index = False)

In [11]:
training_urls = data[1][~data[1]['camera_id'].isin(['Axis-ToroPeak1', 'Axis-Dewdrop1', 'Axis-Berryessa'])]['url'].apply(lambda x: 'https://tools.alertcalifornia.org' + x)

training_labels = majority_labels[majority_labels['image'].isin(training_urls)]
unseen_labels = majority_labels[~majority_labels['image'].isin(training_urls)]

training_labels.to_csv('camera_data/training_set_cameras_data.csv')
unseen_labels.to_csv('camera_data/unseen_set_cameras_data.csv')

In [ ]:
majority[majority_labels['image'].isin(coronado_urls)]

[0        True
 1        True
 2        True
 3        True
 4        True
         ...  
 6946    False
 6947    False
 6948    False
 6949    False
 6950    False
 Name: image, Length: 6951, dtype: bool]

In [8]:
coronado_labels_to_inject.head()

,id,choice
0,4bd763a1-79e2-5a17-9ca0-4c065b09f869.jpg,3 (Major)
1,73e15d36-e9df-5cea-a140-3a1d3856fc00.jpg,2 (Minor)
2,8bb77bee-06a0-55b3-a98c-a30ac4f5202c.jpg,1 (Zero)
3,37142b8a-dea1-5b5c-a210-c99aa4273666.jpg,2 (Minor)
4,c4f156db-b801-5ef7-93d2-a410257c5efe.jpg,2 (Minor)


In [9]:
json = pd.read_json('camera_data/coronado_hills_full.json').head()

In [10]:
from src.dataloading import get_data_urls

labels_csv = 'camera_data/coronado_hills_full.json'
get_data_urls(labels_csv, binarize = False, include_location = False, inject_data = coronado_labels_to_inject).head(-5)['image']

0       https://tools.alertcalifornia.org/sanitized-da...
1       https://tools.alertcalifornia.org/sanitized-da...
2       https://tools.alertcalifornia.org/sanitized-da...
3       https://tools.alertcalifornia.org/sanitized-da...
4       https://tools.alertcalifornia.org/sanitized-da...
                              ...                        
7478    https://tools.alertcalifornia.org/sanitized-da...
7479    https://tools.alertcalifornia.org/sanitized-da...
7480    https://tools.alertcalifornia.org/sanitized-da...
7481    https://tools.alertcalifornia.org/sanitized-da...
7482    https://tools.alertcalifornia.org/sanitized-da...
Name: image, Length: 7483, dtype: str